# DS Automation Assignment

# Task 1
Using our prepared churn data from week 2:
- use pycaret to find an ML algorithm that performs best on the data
    - Choose a metric you think is best to use for finding the best model; by default, it is accuracy but it could be AUC, precision, recall, etc. The week 3 FTE has some information on these different metrics.
- save the model to disk
- create a Python script/file/module with a function that takes a pandas dataframe as an input and returns the probability of churn for each row in the dataframe
    - your Python file/function should print out the predictions for new data (new_churn_data.csv)
    - the true values for the new data are [1, 0, 0, 1, 0] if you're interested
- test your Python module and function with the new data, new_churn_data.csv
- write a short summary of the process and results at the end of this notebook
- upload this Jupyter Notebook and Python file to a Github repository, and turn in a link to the repository in the week 5 assignment dropbox

Additional challenges:
- return the probability of churn for each new prediction, and the percentile where that prediction is in the distribution of probability predictions from the training dataset (e.g. a high probability of churn like 0.78 might be at the 90th percentile)
- use other autoML packages, such as TPOT, H2O, MLBox, etc, and compare performance and features with pycaret
- create a class in your Python module to hold the functions that you created
- accept user input to specify a file using a tool such as Python's `input()` function, the `click` package for command-line arguments, or a GUI
- Use the unmodified churn data (new_unmodified_churn_data.csv) in your Python script. This will require adding the same preprocessing steps from week 2 since this data is like the original unmodified dataset from week 1.


# Task 2: Critical Reflection
What are your observations based on the analyses you had performed? Is there one model that seems particularly useful for predicting churn. Why or why not?

In [8]:
#    build dependencies
from pycaret.classification import setup, compare_models, predict_model, save_model, load_model
import sklearn
from sklearn.naive_bayes import GaussianNB
import pandas as pd
import pickle
from IPython.display import Code


## Use pycaret to find an ML algorithm that performs best on the data
    - Choose a metric you think is best to use for finding the best model; by default, it is accuracy but it could be AUC, precision, recall, etc. The week 3 FTE has some information on these different metrics.

In [9]:
train_data = pd.read_csv('Data/prepared_churn_data.csv', index_col='customerID')
train_data = train_data.drop(columns='AvgMonthly')

#    low importance features identified in the Week 4 assignment
low_importance_features = ['DeviceProtection', 'PaperlessBilling', 'StreamingTV', 'StreamingMovies', 'MultipleLines', 'gender', 'Partner', 'SeniorCitizen', 'Dependents', 'PhoneService']

automl = setup(train_data, target='Churn', ignore_features=low_importance_features, preprocess='true', remove_outliers='true')


,Description,Value
0,Session id,7957
1,Target,Churn
2,Target type,Binary
3,Original data shape,"(7043, 7)"
4,Transformed data shape,"(6796, 6)"
5,Transformed train set shape,"(4683, 6)"
6,Transformed test set shape,"(2113, 6)"
7,Ignore features,10
8,Numeric features,5
9,Preprocess,true


Since only one of the low importance features is part of the training data only one was dropped.  With remove outliers set to true rows were dropped in the data set, train set sizes.  There were 247 rows were removed from the transformed data and train set sizes.  

In [11]:
best_model = compare_models(sort='F1')

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
qda,Quadratic Discriminant Analysis,0.7205,0.8156,0.7775,0.4839,0.5962,0.3999,0.4265,0.0110
nb,Naive Bayes,0.7110,0.8075,0.7630,0.4729,0.5836,0.3806,0.4065,0.0100
ada,Ada Boost Classifier,0.7911,0.8324,0.5099,0.6319,0.5639,0.4287,0.4332,0.0180
lightgbm,Light Gradient Boosting Machine,0.7866,0.8220,0.5137,0.6186,0.5609,0.4216,0.4250,0.6710
gbc,Gradient Boosting Classifier,0.7913,0.8329,0.4992,0.6356,0.5590,0.4249,0.4304,0.0330
lda,Linear Discriminant Analysis,0.7880,0.8170,0.4908,0.6296,0.5511,0.4152,0.4210,0.0110
lr,Logistic Regression,0.7901,0.8283,0.4854,0.6370,0.5506,0.4170,0.4237,0.0130
ridge,Ridge Classifier,0.7886,0.8170,0.4480,0.6469,0.5289,0.3985,0.4099,0.0110
rf,Random Forest Classifier,0.7708,0.7933,0.4808,0.5829,0.5265,0.3772,0.3805,0.0300
et,Extra Trees Classifier,0.7570,0.7750,0.4816,0.5478,0.5120,0.3513,0.3529,0.0450


An interesting note; before restarting the environment the Ridge Classifier and the Gradient Boosting Classifer matched in accuracy and the GBC model had the most categories of highest rating.  Now i am not receiving consistant results on the highest rated models.

In [12]:
print("Training data shape:    ",train_data.iloc[-2:-1].shape,"\n")
predict_model(best_model, train_data.iloc[-2:-1])

Training data shape:     (1, 7) 



,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Quadratic Discriminant Analysis,1.0000,0,1.0000,1.0000,1.0000,nan,0.0000


,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,Churn,prediction_label,prediction_score
customerID,,,,,,,,,
8361-LTMKD,4,1,0,1,74.400002,306.600006,1,1,0.9049


In [13]:
best_model

QuadraticDiscriminantAnalysis(priors=None, reg_param=0.0,
                              store_covariance=False, tol=0.0001)

## Save the model to disk

In [14]:
save_model(best_model, 'Models/QDA')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['tenure', 'Contract',
                                              'PaymentMethod', 'MonthlyCharges',
                                              'TotalCharges'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill_value=None,
                                                               keep_empty_features=False,
                                                               missing_values=nan,
                                                               strategy='mean'))),
                 ('categorical_imputer',
                  TransformerWrap...
                                                               keep_empty_features=Fal

## Create a Python script/file/module with a function that takes a pandas dataframe as an input and returns the probability of churn for each row in the dataframe
    - your Python file/function should print out the predictions for new data (new_churn_data.csv)
    - the true values for the new data are [1, 0, 0, 1, 0] if you're interested

## test your Python module and function with the new data, new_churn_data.csv

In [15]:
Code('predictChurn_v2.py')

import sys
import pandas as pd
from pycaret.classification import predict_model, load_model

class ChurnPredictor:
    #def __init__(self, model_path):
    #    self.model = load_model(model_path)

    def __init__(self, model_path, train_path=None):
        self.model = load_model(model_path)
        self.train_probability = None
        if train_path:
            train_df = self.load_data(train_path)
            train_df = train_df.drop(columns=['Churn', 'AvgMonthly'], errors='ignore')
            self.train_probability = self.get_churn_probability(train_df)

    def predict(self, df):
        predictions = predict_model(self.model, data=df)

        predictions.rename(
            {'prediction_label': 'Churn_prediction'},
            axis=1,
            inplace=True
        )

        predictions['Churn_prediction'].replace(
            {1: 'Churn', 0: 'No churn'},
            inplace=True
        )

        return predictions['Churn_prediction']
    
    def score_predict(self, df):
        predictions = predict_model(self.model, data=df)

        predictions.rename(
            {
                'prediction_label': 'Churn_prediction',
                'prediction_score': 'Churn_probability'
            },
            axis=1,
            inplace=True
        )

        predictions['Churn_prediction'] = predictions['Churn_prediction'].replace(
            {1: 'Churn', 0: 'No churn'}
        )

        return predictions[['Churn_prediction', 'Churn_probability']]


def load_data(filepath):
    return pd.read_csv(filepath, index_col='customerID')


if __name__ == "__main__":
    if len(sys.argv) > 3:
        print("Usage: python predictChurn.py <data_file> <model_path>")
        sys.exit(1)

    data_file = sys.argv[1]
    model_path = sys.argv[2]

    #df = load_data('Data/new_churn_data.csv')
    df = load_data(data_file)

    predictor = ChurnPredictor(model_path)

    if "Ridge" in model_path or "SVM" in model_path:
        predictions = predictor.predict(df)
    else:
        predictions = predictor.score_predict(df)

    print("predictions:")
    print(predictions)

In [16]:
%run predictChurn_v2.py Data/new_churn_data.csv Models/QDA


Transformation Pipeline and Model Successfully Loaded


predictions:
           Churn_prediction  Churn_probability
customerID                                    
9305-CKSKC            Churn             0.8043
1452-KNGVK         No churn             0.9747
6723-OKKJM         No churn             0.9975
7832-POPKP         No churn             1.0000
6348-TACGU         No churn             1.0000


## Upload this Jupyter Notebook and Python file to a Github repository, and turn in a link to the repository in the week 5 assignment dropbox

# Additional challenges:

## return the probability of churn for each new prediction, and the percentile where that prediction is in the distribution of probability predictions from the training dataset (e.g. a high probability of churn like 0.78 might be at the 90th percentile)

## Create a class in your Python module to hold the functions that you created
## Accept user input to specify a file using a tool such as Python's `input()` function, the `click` package for command-line arguments, or a GUI
## Use the unmodified churn data (new_unmodified_churn_data.csv) in your Python script. This will require adding the same preprocessing steps from week 2 since this data is like the original unmodified dataset from week 1.

In [17]:
new_data = pd.read_csv('Data/new_churn_data_unmodified.csv')
print(new_data.isna().sum())

customerID        0
tenure            0
PhoneService      0
Contract          0
PaymentMethod     0
MonthlyCharges    0
TotalCharges      0
dtype: int64


In [18]:
new_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customerID      5 non-null      object 
 1   tenure          5 non-null      int64  
 2   PhoneService    5 non-null      object 
 3   Contract        5 non-null      object 
 4   PaymentMethod   5 non-null      object 
 5   MonthlyCharges  5 non-null      float64
 6   TotalCharges    5 non-null      float64
dtypes: float64(2), int64(1), object(4)
memory usage: 408.0+ bytes


In [19]:
new_data['PhoneService'] = new_data['PhoneService'].map({'No': 0, 'Yes': 1})
new_data['Contract'] = new_data['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
new_data['PaymentMethod'] = new_data['PaymentMethod'].map({'Electronic check': 0, 'Mailed check': 1, 'Bank transfer (automatic)': 2, 'Credit card (automatic)': 3})

new_data.to_csv('Data/new_churn_data_cleaned.csv')

In [217]:
print('Ridge Classifier:')
%run predictChurn_v2.py Data/new_churn_data_cleaned.csv Models/Ridge
print('\nGradient Boosting Classifier:')
%run predictChurn_v2.py Data/new_churn_data_cleaned.csv Models/GBC
print('\nAda Boost Classifier:')
%run predictChurn.py Data/new_churn_data_cleaned.csv Models/ADA
print('\nLogistic Regression:')
%run predictChurn.py Data/new_churn_data_cleaned.csv Models/LR
print('\nNaive Bayes:')
%run predictChurn.py Data/new_churn_data_cleaned.csv Models/NB

Ridge Classifier:
Transformation Pipeline and Model Successfully Loaded


predictions:
customerID
9305-CKSKC       Churn
1452-KNGVK    No churn
6723-OKKJM    No churn
7832-POPKP       Churn
6348-TACGU    No churn
Name: Churn_prediction, dtype: object

Gradient Boosting Classifier:
Transformation Pipeline and Model Successfully Loaded


predictions:
customerID
9305-CKSKC       Churn
1452-KNGVK    No churn
6723-OKKJM    No churn
7832-POPKP    No churn
6348-TACGU    No churn
Name: Churn_prediction, dtype: object

Ada Boost Classifier:
Transformation Pipeline and Model Successfully Loaded


predictions:
customerID
9305-CKSKC       Churn
1452-KNGVK    No churn
6723-OKKJM    No churn
7832-POPKP    No churn
6348-TACGU    No churn
Name: Churn_prediction, dtype: object

Logistic Regression:
Transformation Pipeline and Model Successfully Loaded


predictions:
customerID
9305-CKSKC       Churn
1452-KNGVK    No churn
6723-OKKJM    No churn
7832-POPKP    No churn
6348-TACGU    No churn
Name: Churn_prediction, dtype: object

Naive Bayes:
Transformation Pipeline and Model Successfully Loaded


predictions:
customerID
9305-CKSKC       Churn
1452-KNGVK    No churn
6723-OKKJM    No churn
7832-POPKP    No churn
6348-TACGU    No churn
Name: Churn_prediction, dtype: object


# Task 2: Critical Reflection
What are your observations based on the analyses you had performed? Is there one model that seems particularly useful for predicting churn. Why or why not?

The model comparison has been inconsistent after restarting the environment.  The Ridge Classifier seems to be the most consistent in the comparison for Accuracy and Precision.  The Gradient Boosting Classifier frequently shows the most categories with the highest rating.  I would lean toward using the Ridge Classifier model since it has been the most consistent in the model comparison.  

# Summary
## Including summary of the process and results

In the training data low importance features identified in the week 4 assignment were dropped along with the AvgMonthly feature that was an added feature from week 2.  This data was used to identify a model to then use to predict churn on new data sets.  The model was saved then used in a python script predict churn from the new data sets.  In the additional challenges the new_churn_data_unmodified.csv data was checked for missing data then the non-numeric columns were converted to numeric.  The cleaned data was saved then run through the script to predict churn for the customers included in that data set.

# AI Disclosure

Used Google/Gemini for package methods, method parameters, and syntax.

# Export to HTML

In [219]:
!jupyter nbconvert --to html "Russell_Josh_Assignment5.ipynb"

[NbConvertApp] Converting notebook Russell_Josh_Assignment5.ipynb to html
[NbConvertApp] Writing 366454 bytes to Russell_Josh_Assignment5.html
